# Machines / maintenance — generate a run & explore

Same logic as the other sources, for the **machines/maintenance** source
(SQL dump loaded into SQLite via SQLAlchemy ORM). Reuses `src/sources/machines/`.

- **Section A** reproduces the standard production graphs by calling `src/`.
- **Section B** adds ad-hoc analyses, including a join to the machine referential.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

%matplotlib inline

from src import config
print("Artifacts dir:", config.MACHINES_ARTIFACTS_DIR)

## 1. Generate a fresh run (or pick the latest)

- `GENERATE_NEW_RUN = True` -> run the machines pipeline (4 plots + report).
- `GENERATE_NEW_RUN = False` -> only explore the most recent existing run.

In [ ]:
from src.sources.machines.runner import run_default

GENERATE_NEW_RUN = False  # True = create a new run; False = explore the latest

if GENERATE_NEW_RUN:
    run_dir = run_default()
else:
    runs = sorted(
        p for p in config.MACHINES_ARTIFACTS_DIR.iterdir()
        if p.is_dir() and p.name.isdigit()
    )
    assert runs, "No machines run found. Set GENERATE_NEW_RUN = True to create one."
    run_dir = runs[-1]

RUN_ID = run_dir.name
print("Active run:", RUN_ID)

## 2. Load the data (SQL dump -> SQLite via SQLAlchemy ORM)

`build_engine` creates a local SQLite database from the dump using the ORM
models; we then read the maintenance events and the machine referential.

In [ ]:
from src.sources.machines.loader import (
    build_engine,
    load_maintenance,
    load_machine_referential,
)

engine = build_engine(config.DEFAULT_MACHINES_SQL)
df = load_maintenance(engine)
machines = load_machine_referential(engine)
df.head()

## 3. Dataset overview

In [ ]:
print("Maintenance shape:", df.shape)
df.info()

In [ ]:
df.describe(include="all").T

## 4. Quality metrics (reuse `src/`)

In [ ]:
from src.common.metrics import compute_quality_metrics

compute_quality_metrics(df)

## A. Standard production graphs (via `src/`)

Calls the exact functions used at each run and displays the saved PNGs.
Regenerates the files inside `run_dir`.

In [ ]:
from src.sources.machines import plots

for png_path in plots.plot_all(df, run_dir):
    display(Image(filename=str(png_path)))

## B. Additional inline analyses

Ad-hoc explorations not part of the standard set.

### B.1 Duration by maintenance type

In [ ]:
df.groupby(config.MAINTENANCE_TYPE_COLUMN)[config.MAINTENANCE_DURATION_COLUMN]\
    .agg(["count", "mean", "median", "max"]).round(2)

### B.2 Maintenance count by machine criticality

Joins maintenance events with the **machine referential** (the ORM relationship)
to see whether more critical machines get more maintenance.

In [ ]:
merged = df.merge(
    machines[[config.MACHINE_COLUMN, config.MACHINE_CRITICALITY_COLUMN]],
    on=config.MACHINE_COLUMN,
    how="left",
)
by_crit = (
    merged.groupby(config.MACHINE_CRITICALITY_COLUMN).size()
    .reindex(["LOW", "MEDIUM", "HIGH"])
)
display(by_crit)

ax = by_crit.plot.bar(figsize=(7, 4), color="#937860")
ax.set_xlabel("Criticality")
ax.set_ylabel("Number of maintenance events")
ax.set_title("Maintenance events by machine criticality")
plt.tight_layout()
plt.show()

### B.3 Reactive maintenances linked to an incident

`related_incident_id` links maintenance back to the incidents source — a basis
for future cross-source analysis.

In [ ]:
linked = df[df[config.MAINTENANCE_INCIDENT_COLUMN].notna()]
print(f"Maintenances linked to an incident: {len(linked)} / {len(df)}")
display(
    linked[[
        config.MACHINE_COLUMN,
        config.MAINTENANCE_TYPE_COLUMN,
        config.MAINTENANCE_COMPONENT_COLUMN,
        config.MAINTENANCE_INCIDENT_COLUMN,
    ]].head(10)
)

## C. Promoting an analysis to production

When a Section B cell proves useful, move its logic into
`src/sources/machines/` and wire it into `runner.py` so it becomes a standard
artifact at every run.